# Baselines

Log normalized_return to TensorBoard for comparison:
- Heuristic baselines: 2200 episodes
- SB3 DQN and PPO: training curves

In [1]:
import numpy as np
import json
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from Env import VoltageDistributionEnv
from CustomFeatureExtractor import CustomFeatureExtractor
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv

with open('grid_configuration.json', 'r') as f:
    grid_config = json.load(f)
grid_size = grid_config['gridSize']

In [11]:
# Random Baseline: 2200 episodes - purely random sampling anywhere on grid
writer = SummaryWriter("models/tensorboard/baselines/Random_11x11")
for ep in tqdm(range(2200), desc="Random"):
    env = VoltageDistributionEnv()
    obs, info = env.reset()
    done, total_reward, steps = False, 0, 0
    while not done:
        action = np.random.randint(0, grid_size * grid_size)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1
    writer.add_scalar("rollout/normalized_return", total_reward / steps, ep)
    env.close()
writer.close()

Random:   0%|          | 0/2200 [00:00<?, ?it/s]

Random: 100%|██████████| 2200/2200 [02:25<00:00, 15.12it/s]


In [12]:
# Greedy Baseline: 2200 episodes - probe highest predicted belief
# The Bayesian filter still runs (predict + correct), but we pick argmax of predicted belief
writer = SummaryWriter("models/tensorboard/baselines/Greedy_11x11")
for ep in tqdm(range(2200), desc="Greedy"):
    env = VoltageDistributionEnv()
    obs, info = env.reset()
    done, total_reward, steps = False, 0, 0
    while not done:
        saved_belief = env.belief.belief.copy()
        env.belief.prediction()
        predicted_belief = env.belief.get_belief_distribution()
        env.belief.belief = saved_belief 
        
        action = np.argmax(predicted_belief)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1
    writer.add_scalar("rollout/normalized_return", total_reward / steps, ep)
    env.close()
writer.close()

Greedy: 100%|██████████| 2200/2200 [02:32<00:00, 14.45it/s]


In [13]:
# Entropy-Weighted Sampling: 2200 episodes
# Sample proportionally to local entropy - explores uncertain regions
# Contrasts with greedy (exploitation) by favoring exploration

def compute_local_entropy(belief, grid_size, window=3):
    """Compute local entropy for each position using a sliding window."""
    belief_2d = belief.reshape(grid_size, grid_size)
    local_entropy = np.zeros_like(belief_2d)
    half_w = window // 2
    
    for i in range(grid_size):
        for j in range(grid_size):
            # Extract local window
            i_min, i_max = max(0, i - half_w), min(grid_size, i + half_w + 1)
            j_min, j_max = max(0, j - half_w), min(grid_size, j + half_w + 1)
            local_probs = belief_2d[i_min:i_max, j_min:j_max].flatten()
            
            # Normalize to valid distribution
            local_sum = local_probs.sum()
            if local_sum > 1e-10:
                local_probs = local_probs / local_sum
                # Compute entropy: -sum(p * log(p))
                local_probs = local_probs[local_probs > 1e-10]
                local_entropy[i, j] = -np.sum(local_probs * np.log(local_probs))
            else:
                local_entropy[i, j] = 0
    
    return local_entropy.flatten()

writer = SummaryWriter("models/tensorboard/baselines/EntropyWeighted_11x11")
for ep in tqdm(range(2200), desc="Entropy-Weighted"):
    env = VoltageDistributionEnv()
    obs, info = env.reset()
    done, total_reward, steps = False, 0, 0
    while not done:
        # Get predicted belief
        saved_belief = env.belief.belief.copy()
        env.belief.prediction()
        predicted_belief = env.belief.get_belief_distribution()
        env.belief.belief = saved_belief
        
        # Compute local entropy and sample proportionally
        local_entropy = compute_local_entropy(predicted_belief, grid_size)
        # Add small epsilon to avoid zero probabilities
        sampling_weights = local_entropy + 1e-10
        sampling_weights /= sampling_weights.sum()
        
        action = np.random.choice(grid_size * grid_size, p=sampling_weights)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1
    writer.add_scalar("rollout/normalized_return", total_reward / steps, ep)
    env.close()
writer.close()

Entropy-Weighted: 100%|██████████| 2200/2200 [05:09<00:00,  7.11it/s]


In [2]:
# SB3 DQN with explicit TensorBoard logging (matches RecurrentDQN format)
from stable_baselines3.common.monitor import Monitor

class DQNNormalizedReturnCallback(BaseCallback):
    def __init__(self, tb_writer, verbose=0):
        super().__init__(verbose)
        self.tb_writer = tb_writer
        self.episode_count = 0
    
    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                normalized_return = info['episode']['r'] / info['episode']['l']
                self.tb_writer.add_scalar('rollout/normalized_return', normalized_return, self.episode_count)
                self.episode_count += 1
        return True

dqn_writer = SummaryWriter("models/tensorboard/baselines/DQN_11x11_PT2")
dqn_env = DummyVecEnv([lambda: Monitor(VoltageDistributionEnv())])

dqn = DQN(
    "MultiInputPolicy",
    dqn_env,
    policy_kwargs=dict(features_extractor_class=CustomFeatureExtractor, features_extractor_kwargs=dict(features_dim=128)),
    learning_rate=1e-4,
    buffer_size=1000,
    batch_size=32,
    gamma=0.99,
    exploration_fraction=0.05,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.001,
    target_update_interval=500,
    verbose=1,
)
dqn.learn(total_timesteps=100_000, callback=DQNNormalizedReturnCallback(dqn_writer))
dqn_writer.close()

Using cpu device
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 36.5     |
|    ep_rew_mean      | 1.17     |
|    exploration_rate | 0.971    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 910      |
|    time_elapsed     | 0        |
|    total_timesteps  | 146      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0284   |
|    n_updates        | 11       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 31.2     |
|    ep_rew_mean      | 1.21     |
|    exploration_rate | 0.95     |
| time/               |          |
|    episodes         | 8        |
|    fps              | 742      |
|    time_elapsed     | 0        |
|    total_timesteps  | 250      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0379   |
|  

In [3]:
# SB3 PPO with explicit TensorBoard logging (matches RecurrentPPO format)
class PPONormalizedReturnCallback(BaseCallback):
    def __init__(self, tb_writer, verbose=0):
        super().__init__(verbose)
        self.tb_writer = tb_writer
        self.episode_count = 0
    
    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                normalized_return = info['episode']['r'] / info['episode']['l']
                self.tb_writer.add_scalar('rollout/normalized_return', normalized_return, self.episode_count)
                self.episode_count += 1
        return True

ppo_writer = SummaryWriter("models/tensorboard/baselines/PPO_11x11_PT2")
ppo_env = DummyVecEnv([lambda: Monitor(VoltageDistributionEnv())])

ppo = PPO(
    "MultiInputPolicy",
    ppo_env,
    policy_kwargs=dict(features_extractor_class=CustomFeatureExtractor, features_extractor_kwargs=dict(features_dim=128)),
    learning_rate=1e-4,
    n_steps=128,
    batch_size=32,
    gamma=0.99,
    verbose=1,
)
ppo.learn(total_timesteps=100_000, callback=PPONormalizedReturnCallback(ppo_writer))
ppo_writer.close()

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 26.5     |
|    ep_rew_mean     | 1.57     |
| time/              |          |
|    fps             | 947      |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 128      |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 31           |
|    ep_rew_mean          | 1.56         |
| time/                   |              |
|    fps                  | 569          |
|    iterations           | 2            |
|    time_elapsed         | 0            |
|    total_timesteps      | 256          |
| train/                  |              |
|    approx_kl            | 0.0043269964 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -4.79        |
|    explained_variance   | -0.0394      